# Fine-Tuning Laya on IT Operations Ticket Complexity (`itops_agent`)

[![Model Architecture](https://img.shields.io/badge/Model-Laya%20(421M)-blue)](https://huggingface.co/convaiinnovations/laya)
[![Objective](https://img.shields.io/badge/Objective-RLCD%20Policy%20Gradient-green)](https://github.com/convai-innovations/laya)
[![Hardware](https://img.shields.io/badge/Hardware-Kaggle%202xT4%20DDP%20%2F%20Local%201xGPU-orange)]()

This notebook fine-tunes **Laya** (`convaiinnovations/laya`, 421M parameters) to classify incoming IT support tickets into three operational execution complexities:
1. **`single_hop`**: Straightforward, deterministic action with no diagnostics required (e.g., password reset, standard software install, license provisioning).
2. **`multi_hop`**: Sequential troubleshooting or diagnostic investigation requiring dependent reasoning steps (e.g., intermittent VPN drops, system service failure, DB connection timeouts).
3. **`parallel_hop`**: Multiple independent operational tasks that can be fulfilled concurrently (e.g., multi-user onboarding, dual software + hardware requests).

### Architecture & Training Highlights:
- **Calibrated Decisions**: Laya uses RLCD (Reinforcement Learning from Calibrated Decisions) with proper scoring rules (Brier loss) to output truly calibrated confidence scores alongside class predictions.
- **Single-Pass Inference**: Evaluates ticket text and the typed decision schema in a single fast forward pass (~10–30 ms latency).
- **Dual Environment Compatible**: Runs out-of-the-box on either **Kaggle Dual T4 GPUs** using Distributed Data Parallel (`torchrun --nproc_per_node=2`) or locally on a **single GPU** (e.g., RTX 4060 Laptop GPU).

## 1. Environment & GPU Verification
Verify available GPU hardware. The notebook automatically supports single-GPU (local) or multi-GPU DDP (Kaggle 2×T4).

In [ ]:
!nvidia-smi
import os, subprocess, torch

n_gpu = torch.cuda.device_count()
print(f"CUDA Available: {torch.cuda.is_available()} | Visible GPUs: {n_gpu}")
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name} ({p.total_memory / 1e9:.1f} GB)")

assert n_gpu >= 1, (
    f"Expected at least 1 GPU, but detected {n_gpu}!\n"
    "If running on Kaggle, select GPU T4 x2 (or GPU P100) under Notebook options -> Accelerator."
)

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print(f"Hardware verified: {n_gpu} GPU(s) available for training.")

## 2. Install Dependencies
Install `laya`, `transformers`, `datasets`, `scikit-learn`, and distributed training utilities.

In [ ]:
!pip install -q -U "laya>=0.1.6" "transformers>=4.48.0" "datasets>=3.0.0" safetensors huggingface_hub pyarrow pandas scipy accelerate tabulate scikit-learn
import laya, transformers, datasets, torch, sklearn
print("Laya version        :", laya.__version__)
print("Transformers version:", transformers.__version__)
print("PyTorch version     :", torch.__version__)
print("Scikit-Learn version:", sklearn.__version__)

## 3. Load & Preprocess IT Support Ticket Dataset
We load `annotated_dataset.csv`, extract labeled tickets, perform a 90/10 stratified split, format rows into Laya's `{state, questions, gold}` decision schema, and tokenize them with `items_from_rows`.

In [ ]:
import os, sys, glob, json, torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer
from huggingface_hub import snapshot_download
from laya.agent import _fix_tokenizer_config
from laya.train import items_from_rows

# Define environment working directory
WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle") else "./laya_work"
os.makedirs(WORK_DIR, exist_ok=True)
print(f"Working directory: {WORK_DIR}")

# Locate dataset CSV
candidate_paths = [
    "src/itops_agent/classifier/dataset/annotated_dataset.csv",
    "../dataset/annotated_dataset.csv",
    "./annotated_dataset.csv",
    "/kaggle/input/itops-complexity/annotated_dataset.csv",
    "/kaggle/input/annotated-dataset/annotated_dataset.csv",
    "/kaggle/input/itops-agent/annotated_dataset.csv"
]
csv_path = None
for p in candidate_paths:
    if os.path.exists(p):
        csv_path = p
        break

if csv_path is None:
    found = glob.glob("**/annotated_dataset.csv", recursive=True)
    if found:
        csv_path = found[0]
    else:
        raise FileNotFoundError(
            "Could not locate 'annotated_dataset.csv'! "
            "Please ensure the dataset file is uploaded or placed in the working directory."
        )

print(f"Loading dataset from: {csv_path}")
df = pd.read_csv(csv_path)
print(f"Total rows in CSV: {len(df):,}")

# Filter for rows that have complexity annotations
valid_classes = ["single_hop", "multi_hop", "parallel_hop"]
df_labeled = df[df["complexity"].isin(valid_classes)].copy().reset_index(drop=True)
print(f"Labeled rows available for fine-tuning: {len(df_labeled):,}")
print("\nClass distribution:")
for cls_name, count in df_labeled["complexity"].value_counts().items():
    pct = count / len(df_labeled) * 100
    print(f"  - {cls_name:12s}: {count:5d} ({pct:5.2f}%)")

# Download base Laya model config and tokenizer
MODEL_ID = "convaiinnovations/laya"
print(f"\nFetching tokenizer and config from {MODEL_ID}...")
model_dir = snapshot_download(MODEL_ID)
_fix_tokenizer_config(model_dir)

tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
    cfg = json.load(f)

# Define ITOps Operational Complexity Question Schema
COMPLEXITY_QUESTION = {
    "complexity": {
        "type": "choice",
        "instructions": "What is the operational complexity required to fulfill this IT support ticket?",
        "criteria": {
            "single_hop": "Self-contained deterministic action with no diagnostic investigation required.",
            "multi_hop": "Sequential troubleshooting, diagnostic investigation, or dependent actions.",
            "parallel_hop": "At least two independent operational tasks executed concurrently."
        }
    }
}

# Perform 90/10 stratified split
# The 90% train set will be split 90/10 inside train_ddp into parameter updates and calibration holdout
df_train, df_test = train_test_split(
    df_labeled,
    test_size=0.10,
    random_state=42,
    stratify=df_labeled["complexity"]
)
print(f"\nSplit: {len(df_train)} train/calib rows, {len(df_test)} benchmark test rows.")

def row_to_laya_format(row, idx):
    comp = str(row["complexity"])
    return {
        "id": f"ticket_{idx}",
        "state": {
            "type": str(row["type"]) if pd.notna(row.get("type")) else "Generic",
            "ticket": str(row["ticket"])
        },
        "questions": COMPLEXITY_QUESTION,
        "gold": {
            "complexity": {
                "label": comp,
                "probabilities": {
                    "single_hop": 1.0 if comp == "single_hop" else 0.0,
                    "multi_hop": 1.0 if comp == "multi_hop" else 0.0,
                    "parallel_hop": 1.0 if comp == "parallel_hop" else 0.0
                }
            }
        }
    }

train_rows = [row_to_laya_format(row, i) for i, (_, row) in enumerate(df_train.iterrows())]
test_cases = [row_to_laya_format(row, i) for i, (_, row) in enumerate(df_test.iterrows())]

# Tokenize training items
train_items, skipped = items_from_rows(tok, train_rows, cfg["max_len"], cfg["head_max_len"])
print(f"Preprocessed {len(train_items)} training items (skipped {skipped}).")

# Save preprocessed train items and test cases
train_items_path = os.path.join(WORK_DIR, "train_items.pt")
test_cases_path = os.path.join(WORK_DIR, "test_cases.json")

torch.save(train_items, train_items_path)
with open(test_cases_path, "w") as f:
    json.dump(test_cases, f, indent=2)

print(f"Saved preprocessed training items to: {train_items_path}")
print(f"Saved benchmark test cases to: {test_cases_path}")

## 4. Distributed Training Script (`train_ddp.py`)
We write the distributed RLCD training script. It handles:
- Multi-GPU (Kaggle 2xT4) or single-GPU training with gradient synchronization.
- Sigma-annealed exploration with proper scoring rules (Brier loss).
- Automatic calibration holdout split (`split_calibration`).
- Post-training temperature calibration fitting (`fit_temperature_map`).

In [ ]:
%%writefile train_ddp.py
"""DDP training script for Laya IT Operations ticket complexity fine-tuning.
Optimized for single-GPU 8GB VRAM (RTX 4060) and multi-GPU (2xT4).
"""
import os, sys, time, json, random, math
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import numpy as np
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from safetensors.torch import load_file
from transformers import AutoTokenizer

from laya.common import build_model, collate_items
from laya.calibrate import fit_temperature_map
from laya.train import (
    calibration_records,
    encode_item,
    rlcd_loss,
    save_checkpoint,
    sigma_at,
    split_calibration,
)

def main():
    dist.init_process_group("nccl")
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank)

    model_dir = sys.argv[1]
    output_dir = sys.argv[2]
    train_items_path = sys.argv[3] if len(sys.argv) > 3 else "train_items.pt"

    with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
        cfg = json.load(f)
    cfg["gradient_checkpointing"] = True
    cfg["max_tokens_per_batch"] = 2048
    cfg["max_len"] = 512
    cfg["head_max_len"] = 192
    max_len = cfg["max_len"]
    head_max_len = cfg["head_max_len"]

    tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
    model = build_model(cfg, encoder_dir=os.path.join(model_dir, "encoder"))

    weights = load_file(os.path.join(model_dir, "model.safetensors"))
    model.load_state_dict(weights, strict=True)

    model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.head_checkpointing = True
    model.to(device)
    model.train()

    ddp_model = DDP(model, device_ids=[local_rank], find_unused_parameters=True)

    all_items = torch.load(train_items_path, weights_only=False)

    train_items, calib_items = split_calibration(all_items, calib_max=400, calib_frac=0.1, seed=20260922)
    train_items = train_items[:len(train_items) // world_size * world_size]
    my_items = train_items[rank::world_size]

    EPOCHS = 4
    MICRO_BATCH = 4
    GRAD_ACCUM = max(1, 16 // world_size)  # 16 for 1 GPU -> 4 * 16 = 64 effective batch size
    GROUP_SIZE = 4
    LR_ENCODER = 2.5e-5
    LR_HEAD = 1.0e-4
    SIGMA_START = 0.4
    SIGMA_END = 0.1

    enc_params = [p for n, p in ddp_model.named_parameters() if "encoder." in n]
    head_params = [p for n, p in ddp_model.named_parameters() if "encoder." not in n]

    # Use 8-bit AdamW if available to save ~2.5 GB VRAM on 8GB GPUs
    try:
        import bitsandbytes as bnb
        optimizer_cls = bnb.optim.AdamW8bit
        if rank == 0:
            print("Optimizer: bitsandbytes 8-bit AdamW (saving 2.5GB VRAM)")
    except Exception:
        optimizer_cls = torch.optim.AdamW
        if rank == 0:
            print("Optimizer: standard torch.optim.AdamW")

    optimizer = optimizer_cls([
        {"params": enc_params, "lr": LR_ENCODER},
        {"params": head_params, "lr": LR_HEAD}
    ], weight_decay=0.01)

    total_updates = (len(my_items) // (MICRO_BATCH * GRAD_ACCUM)) * EPOCHS
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, total_updates), eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda", enabled=True)

    if rank == 0:
        print(f"Starting training on {torch.cuda.get_device_name(device)}:")
        print(f"  - {len(train_items)} train items ({len(calib_items)} calibration holdout)")
        print(f"  - Micro-batch: {MICRO_BATCH} | Grad Accum: {GRAD_ACCUM} | Effective batch: 64")
        print(f"  - {EPOCHS} epochs | {total_updates} total optimizer updates")
    t0 = time.time()

    for epoch in range(EPOCHS):
        random.seed(42 + epoch + rank)
        random.shuffle(my_items)
        epoch_loss, n_batches = 0.0, 0
        optimizer.zero_grad(set_to_none=True)
        accum_step = 0

        sigma = sigma_at(epoch, EPOCHS, SIGMA_START, SIGMA_END)
        steps_per_epoch = math.ceil(len(my_items) / MICRO_BATCH)

        for b_idx in range(0, len(my_items), MICRO_BATCH):
            chunk = my_items[b_idx:b_idx + MICRO_BATCH]
            if not chunk:
                continue

            encoded = [encode_item(tok, it, max_len, head_max_len) for it in chunk]
            batch = collate_items([encoded], tok.pad_token_id)

            with torch.autocast("cuda", dtype=torch.float16):
                logits, act = ddp_model(
                    batch["input_ids"].to(device),
                    batch["attention_mask"].to(device),
                    batch["marker_pos"].to(device),
                    batch["marker_mask"].to(device),
                    batch["qtype"].to(device)
                )

            logits = logits.float()
            mask = batch["marker_mask"].to(device)
            target = batch["target"].to(device)

            loss = rlcd_loss(logits, target, mask, batch["qtype"].to(device), sigma,
                             samples=GROUP_SIZE, w_sph=0.75, w_rps=1.0)

            window_start = (accum_step // GRAD_ACCUM) * GRAD_ACCUM
            window_size = min(GRAD_ACCUM, steps_per_epoch - window_start)
            scaled = loss / window_size
            scaler.scale(scaled).backward()
            accum_step += 1

            if accum_step % GRAD_ACCUM == 0 or (b_idx + MICRO_BATCH) >= len(my_items):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(ddp_model.parameters(), 1.0)
                scale_before = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                scale_after = scaler.get_scale()
                if scale_before <= scale_after:
                    scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            epoch_loss += loss.item()
            n_batches += 1

            if rank == 0 and (n_batches % 50) == 0:
                cur_lr = scheduler.get_last_lr()[0]
                print(f"  Epoch {epoch+1}/{EPOCHS} | Step {n_batches}/{steps_per_epoch} | Loss: {loss.item():.4f} | LR: {cur_lr:.2e}")

        if rank == 0:
            print(f"=== Epoch {epoch+1}/{EPOCHS} Completed in {time.time()-t0:.1f}s | Avg Loss: {epoch_loss/max(1, n_batches):.4f} ===")

        dist.barrier()

    dist.barrier()

    # Post-training temperature calibration on rank 0
    if rank == 0:
        print("\n5. Fitting post-training calibration temperatures...")
        del optimizer, scaler, scheduler
        torch.cuda.empty_cache()
        model.eval()
        try:
            records = calibration_records(model, tok, calib_items, device, max_len, head_max_len, batch_size=8)
            fitted = fit_temperature_map(records)
            fitted_temps = fitted["temperature"]
            print("Fitted calibration temperatures:", [round(t, 3) for t in fitted_temps])
        except Exception as e:
            print("Temperature fitting fallback:", e)
            fitted_temps, fitted = None, {"temperature_by_options": {}}

        out_cfg = dict(cfg, max_len=max_len, head_max_len=head_max_len)
        out_cfg["fine_tuned"] = True
        out_cfg["model_name"] = "laya-itops-complexity"
        if fitted_temps is not None:
            out_cfg["temperature"] = fitted_temps
            out_cfg.pop("temperature_by_options", None)
            if fitted["temperature_by_options"]:
                out_cfg["temperature_by_options"] = fitted["temperature_by_options"]
        save_checkpoint(model, tok, out_cfg, output_dir)
        print(f"\nModel successfully saved to: {output_dir}")

    dist.destroy_process_group()

if __name__ == "__main__":
    main()


## 5. Launch Fine-Tuning with `torchrun`
Launch distributed policy-gradient training. It runs on all available GPUs in parallel (~4 to 8 minutes total).

In [ ]:
OUTPUT_DIR = os.path.join(WORK_DIR, "laya_finetuned_itops_complexity")
MODEL_DIR = model_dir
nproc = max(1, torch.cuda.device_count())

train_script = "train_ddp.py"
cmd = f"torchrun --standalone --nproc_per_node={nproc} {train_script} {MODEL_DIR} {OUTPUT_DIR} {train_items_path}"
print(f"Executing training with torchrun (nproc_per_node={nproc})...")
print(f"Command: {cmd}\n")
!{cmd}

## 6. Run Benchmark Evaluation on Holdout Test Split
Evaluate the fine-tuned model against the un-seen holdout test cases (10% stratified test split).

In [ ]:
import time, json
import numpy as np
import pandas as pd
import laya
from laya.common import ece_score

print(f"Loading test cases from {test_cases_path}...")
with open(test_cases_path) as f:
    test_cases = json.load(f)

agent_ft = laya.Agent(OUTPUT_DIR, device="cuda")

predictions = []
latencies_ms = []

print(f"Evaluating {len(test_cases)} holdout test cases on GPU...")
t0_eval = time.time()

for i, case in enumerate(test_cases):
    case_id = case["id"]
    state = case["state"]
    questions = case["questions"]
    gold = case["gold"]
    
    t0 = time.perf_counter()
    res = agent_ft.predict(state, questions)
    dt_ms = (time.perf_counter() - t0) * 1000
    latencies_ms.append(dt_ms)
    
    predictions.append({
        "id": case_id,
        "type": state.get("type", "unknown"),
        "pred": res["answers"],
        "gold": gold,
        "questions": questions,
        "latency_ms": dt_ms
    })

print(f"Evaluated all {len(test_cases)} test cases in {time.time() - t0_eval:.1f}s!")

## 7. Compute Official Metrics, Calibration & Latency Benchmarks
Calculate classification accuracy, expected calibration error (ECE), Brier score, inference latency percentiles, and generate the classification report and confusion matrix.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
from laya.common import ece_score

y_true = []
y_pred = []
accuracies = []
soft_accuracies = []
brier_scores = []
kl_divs = []
tv_distances = []
all_confs = []
all_corrects = []

valid_classes = ["single_hop", "multi_hop", "parallel_hop"]

for item in predictions:
    pred_answers = item["pred"]
    gold_answers = item["gold"]
    questions = item["questions"]
    
    qid = "complexity"
    p_ans = pred_answers[qid]
    g_ans = gold_answers[qid]
    qdef = questions[qid]
    
    pred_choice = str(p_ans["choice"])
    gold_label = str(g_ans["label"])
    
    y_true.append(gold_label)
    y_pred.append(pred_choice)
    
    is_corr = float(pred_choice == gold_label)
    accuracies.append(is_corr)
    all_corrects.append(is_corr)
    
    p_probs = np.array([p_ans["probabilities"].get(k, 1e-6) for k in valid_classes])
    g_probs = np.array([g_ans["probabilities"].get(k, 1e-6) for k in valid_classes])
    p_probs /= p_probs.sum()
    g_probs /= g_probs.sum()
    
    all_confs.append(float(p_probs.max()))
    soft_accuracies.append(float((p_probs * g_probs).sum()))
    brier_scores.append(float(((p_probs - g_probs) ** 2).sum()))
    tv_distances.append(float(0.5 * np.abs(p_probs - g_probs).sum()))
    kl_divs.append(float((g_probs * np.log(np.clip(g_probs / p_probs, 1e-12, 1e4))).sum()))

laya_acc = np.mean(accuracies)
laya_soft_acc = np.mean(soft_accuracies)
laya_brier = np.mean(brier_scores)
laya_kl = np.mean(kl_divs)
laya_tv = np.mean(tv_distances)
laya_ece = ece_score(np.array(all_confs), np.array(all_corrects))
laya_latency_p50 = np.percentile(latencies_ms, 50)
laya_latency_p95 = np.percentile(latencies_ms, 95)

# Classification Report
print("=== CLASSIFICATION REPORT (ITOps Operational Complexity) ===\n")
report_dict = classification_report(y_true, y_pred, labels=valid_classes, output_dict=True)
print(classification_report(y_true, y_pred, labels=valid_classes, digits=4))

# Confusion Matrix
print("\n=== CONFUSION MATRIX ===")
cm = confusion_matrix(y_true, y_pred, labels=valid_classes)
df_cm = pd.DataFrame(cm, index=[f"True: {c}" for c in valid_classes], columns=[f"Pred: {c}" for c in valid_classes])
print(df_cm.to_string())

# Benchmark Comparison Table
majority_acc = (np.array(y_true) == "multi_hop").mean()

comparison_data = [
    {
        "Model": "Majority Class Baseline (multi_hop)",
        "Kind": "baseline",
        "Accuracy": round(float(majority_acc), 3),
        "Soft Acc": "-",
        "Brier": "-",
        "ECE": "-",
        "Latency (p50)": "-",
        "Cost/Case": "$0.00"
    },
    {
        "Model": "Laya (Fine-Tuned 421M)",
        "Kind": "specialist",
        "Accuracy": round(float(laya_acc), 3),
        "Soft Acc": round(float(laya_soft_acc), 3),
        "Brier": round(float(laya_brier), 3),
        "ECE": round(float(laya_ece), 3),
        "Latency (p50)": f"{laya_latency_p50:.1f} ms",
        "Cost/Case": "$0.00 (Self-Hosted)"
    },
    {
        "Model": "DeepSeek-R1 (8B) Teacher Ceiling",
        "Kind": "teacher",
        "Accuracy": 1.000,
        "Soft Acc": "-",
        "Brier": "-",
        "ECE": "-",
        "Latency (p50)": "~8,500 ms",
        "Cost/Case": "High Compute / Local Ollama"
    }
]

df_comp = pd.DataFrame(comparison_data)
print("\n=== HEAD-TO-HEAD BENCHMARK TABLE ===\n")
print(df_comp.to_markdown(index=False))

## 8. (Optional) Push Fine-Tuned Model to Hugging Face Hub
Upload the fine-tuned checkpoint, tokenizer, and auto-generated model card to Hugging Face Hub.

In [ ]:
import os
from huggingface_hub import HfApi

# Retrieve HF_TOKEN from environment or Kaggle Secrets
token = os.environ.get("HF_TOKEN")
if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        token = None

if not token:
    print("HF_TOKEN not found. Skipping Hugging Face upload.")
    print("To upload, set HF_TOKEN as an environment variable or in Kaggle Secrets.")
else:
    NEW_REPO = "convaiinnovations/laya-itops-complexity"
    print(f"Publishing fine-tuned model ({laya_acc:.3f} accuracy) to {NEW_REPO}...")

    readme_content = f"""---
license: apache-2.0
library_name: transformers
tags:
- laya
- itops
- ticket-classification
- operational-complexity
- calibrated-decisions
- rlcd
metrics:
- accuracy
- brier_score
model-index:
- name: laya-itops-complexity
  results:
  - task:
      type: text-classification
      name: IT Operations Complexity Classification
    dataset:
      type: itops-agent/annotated_dataset
      name: ITOps Ticket Complexity Dataset
    metrics:
    - type: accuracy
      value: {laya_acc:.3f}
    - type: brier_score
      value: {laya_brier:.3f}
---

# Laya (Fine-Tuned on IT Operations Ticket Complexity)

This model is **Laya** (421M parameters) fine-tuned on IT support tickets to predict operational execution complexity (`single_hop`, `multi_hop`, `parallel_hop`).

Ground-truth supervisory signals were curated using **DeepSeek-R1 (8B)** reasoning guidelines for deterministic single-hop execution, diagnostic multi-hop investigation, and concurrent parallel-hop operations.

## Evaluation Results (Holdout Test Set)

| Metric | Value |
|---|---|
| **Accuracy** | **{laya_acc:.4f}** |
| **Soft Accuracy** | **{laya_soft_acc:.4f}** |
| **Brier Score** | **{laya_brier:.4f}** |
| **Expected Calibration Error (ECE)** | **{laya_ece:.4f}** |
| **Inference Latency (p50)** | **{laya_latency_p50:.1f} ms** |
| **Inference Latency (p95)** | **{laya_latency_p95:.1f} ms** |

## Quickstart

```python
import laya

# Load the fine-tuned model
agent = laya.load("{NEW_REPO}")

# Predict operational complexity of a ticket
state = {{
    "type": "Incident",
    "ticket": "VPN connection drops every 10 minutes when connected to office gateway."
}}

questions = {{
    "complexity": {{
        "type": "choice",
        "instructions": "What is the operational complexity required to fulfill this IT support ticket?",
        "criteria": {{
            "single_hop": "Self-contained deterministic action with no diagnostic investigation required.",
            "multi_hop": "Sequential troubleshooting, diagnostic investigation, or dependent actions.",
            "parallel_hop": "At least two independent operational tasks executed concurrently."
        }}
    }}
}}

result = agent.predict(state, questions)
print("Predicted Complexity:", result["answers"]["complexity"]["choice"])
print("Calibrated Probabilities:", result["answers"]["complexity"]["probabilities"])
```
"""

    with open(os.path.join(OUTPUT_DIR, "README.md"), "w") as f:
        f.write(readme_content)

    print(f"Generated model card at {os.path.join(OUTPUT_DIR, 'README.md')}")

    api = HfApi(token=token)
    api.create_repo(NEW_REPO, repo_type="model", private=False, exist_ok=True)

    api.upload_folder(
        folder_path=OUTPUT_DIR,
        repo_id=NEW_REPO,
        repo_type="model",
        commit_message=f"Laya fine-tuned on ITOps complexity: Acc {laya_acc:.4f}, Brier {laya_brier:.4f}"
    )

    print(f"\nModel successfully published to: https://huggingface.co/{NEW_REPO}")

## 9. Save Benchmark Report as JSON
Export the comprehensive evaluation metrics, classification breakdown, confusion matrix, and latency profile to JSON.

In [ ]:
import json, os

report = {
    "benchmark": "itops-agent/ticket-complexity",
    "model": "Laya-ITOps-Complexity (Fine-Tuned 421M)",
    "n_test_cases": len(test_cases),
    "classes": valid_classes,
    "metrics": {
        "accuracy": round(float(laya_acc), 4),
        "soft_accuracy": round(float(laya_soft_acc), 4),
        "brier_score": round(float(laya_brier), 4),
        "ece": round(float(laya_ece), 4),
        "total_variation": round(float(laya_tv), 4),
        "kl_divergence": round(float(laya_kl), 4),
        "latency_p50_ms": round(float(laya_latency_p50), 1),
        "latency_p95_ms": round(float(laya_latency_p95), 1),
        "macro_f1": round(float(report_dict["macro avg"]["f1-score"]), 4),
        "weighted_f1": round(float(report_dict["weighted avg"]["f1-score"]), 4)
    },
    "per_class": {
        cls: {
            "precision": round(float(report_dict[cls]["precision"]), 4),
            "recall": round(float(report_dict[cls]["recall"]), 4),
            "f1_score": round(float(report_dict[cls]["f1-score"]), 4),
            "support": int(report_dict[cls]["support"])
        }
        for cls in valid_classes if cls in report_dict
    },
    "confusion_matrix": cm.tolist(),
    "per_ticket_type": {}
}

# Calculate per-ticket-type accuracy
ticket_types = set(item["type"] for item in predictions)
for t_type in ticket_types:
    type_items = [item for item in predictions if item["type"] == t_type]
    type_corr = [
        float(item["pred"]["complexity"]["choice"] == str(item["gold"]["complexity"]["label"]))
        for item in type_items
    ]
    if type_corr:
        report["per_ticket_type"][str(t_type)] = {
            "n_cases": len(type_corr),
            "accuracy": round(float(np.mean(type_corr)), 4)
        }

out_file1 = os.path.join(WORK_DIR, "itops_laya_benchmark_report.json")
out_file2 = os.path.join(OUTPUT_DIR, "benchmark_report.json")

with open(out_file1, "w") as f:
    json.dump(report, f, indent=2)

with open(out_file2, "w") as f:
    json.dump(report, f, indent=2)

print(f"Benchmark results successfully saved to:")
print(f"  - {out_file1}")
print(f"  - {out_file2}")
print("\nGlobal Metrics Summary:")
print(json.dumps(report["metrics"], indent=2))
print("\nPer-Class Breakdown:")
print(json.dumps(report["per_class"], indent=2))